#### **TF-IDF**

Defining the TF-IDF class

In [1]:
from src.retrievers.tf_idf import TFIDF
from src.retrievers.base_retriever import Retriever

class DevTFIDF(TFIDF):
    def __init__(self, documents):
        Retriever.__init__(self, documents)   # builds self.index: DocumentDataset

        # these 3 are used only to call GetDocumentsTFIDFVectors. 
        # everywhere else, we just use the methods directly.
        self.corpus_vocabulary = self.GetCorpusVocabulary()
        self.idf = self.GetInverseDocumentFrequencies()
        self.tfidf_matrix = self.GetDocumentsTFIDFVectors()

Instantiating a dummy dataset (of 3 dummy documents) just for testing in this notebook. 

* Note the difference between `raw_docs` and `docs` is the `DocumentDataset` class. 

* The `raw_docs` are not wrapped with the `DocumentDataset` class.

* This happens when passed to the `Retriever` class, in `__BuildIndex`, which we can call using `.index` on the TFIDF `Retriever`.

In [2]:
raw_docs = [{
    'text': ['the cat sat on the mat', 'the dog sat on the log', 'cats and dogs'],
    'document_id': ['d0', 'd1', 'd2']}]

Instantiating the TF-IDF retriever class for testing utility functions (from `src.retrievers.tf_idf`) in this notebook

* Notice the `TFIDF` class takes the `raw_docs` as input.
   
* `class DevTFIDF(documents: list[dict[str, list[str]]])`

In [3]:
tf_idf = DevTFIDF(raw_docs) # instantiate the class (same way as in main.py), to access the methods from TFIDF retriever

To access the document objects we use the `docs` directly as this was built using the `DocumentDataset` class.
* `(variable) docs: Any | DocumentDataset`

In [4]:
docs = tf_idf.index # the DocumentDataset the retriever built
docs.documents[0], docs.documents[1], docs.documents[2]

(<src.data_utils.document.Document at 0x110da4e60>,
 <src.data_utils.document.Document at 0x110da4ec0>)

To access the attributes of each `Document` object directly (text, document_id), we use `docs`

In [5]:
print(docs.documents[0].GetText())
print(docs.documents[0].GetId())

the cat sat on the mat
d0


For example, to get the vocabulary or term counts of a document:

In [6]:
print(tf_idf.GetDocumentVocabulary(docs.documents[0]))
print(tf_idf.GetDocumentTermCounts(docs.documents[0]))

['cat', 'mat', 'on', 'sat', 'the']
{'cat': 1, 'mat': 1, 'on': 1, 'sat': 1, 'the': 2}


To access the methods of the `TFIDF` class instance, we use the `raw_docs`.

Preprocess text (used directly on any string)

In [7]:
print(tf_idf.PreprocessText(raw_docs[0]["text"][0]))
print(tf_idf.PreprocessText(raw_docs[0]["document_id"][0]))

['the', 'cat', 'sat', 'on', 'the', 'mat']
['d0']


Compute the *Document Frequencies* and *Inverse Document Frequencies* for all `raw_docs`

In [8]:
print(tf_idf.GetDocumentFrequencies()) # tf_idf = DevTFIDF(raw_docs) was defined earlier, so tf_idf already has raw_docs
print(tf_idf.GetInverseDocumentFrequencies())

{'cat': 1, 'mat': 1, 'on': 2, 'sat': 2, 'the': 2, 'dog': 1, 'log': 1, 'and': 1, 'cats': 1, 'dogs': 1}
{'cat': np.float64(1.0986122886681098), 'mat': np.float64(1.0986122886681098), 'on': np.float64(0.4054651081081644), 'sat': np.float64(0.4054651081081644), 'the': np.float64(0.4054651081081644), 'dog': np.float64(1.0986122886681098), 'log': np.float64(1.0986122886681098), 'and': np.float64(1.0986122886681098), 'cats': np.float64(1.0986122886681098), 'dogs': np.float64(1.0986122886681098)}


Compute the TFIDF vectors.

* `GetDocumentsTFIDFVector` constructs a *sparse* matrix (zero values are not shown) which includes the TF-IDF scores. 

* Specifically, it returns a n_documents x n_terms matrix whose element (i, j) is the tf-idf score of the j'th term in the i'th document.

In [9]:
for idx, word in enumerate(tf_idf.GetCorpusVocabulary()):
    print(f'{idx} : {word}')

0 : and
1 : cat
2 : cats
3 : dog
4 : dogs
5 : log
6 : mat
7 : on
8 : sat
9 : the


In [10]:
print(tf_idf.GetDocumentsTFIDFVectors())

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 13 stored elements and shape (3, 10)>
  Coords	Values
  (0, 1)	0.5493061542510986
  (0, 6)	0.5493061542510986
  (0, 7)	0.20273254811763763
  (0, 8)	0.20273254811763763
  (0, 9)	0.40546509623527527
  (1, 3)	0.5493061542510986
  (1, 5)	0.5493061542510986
  (1, 7)	0.20273254811763763
  (1, 8)	0.20273254811763763
  (1, 9)	0.40546509623527527
  (2, 0)	1.0986123085021973
  (2, 2)	1.0986123085021973
  (2, 4)	1.0986123085021973


So this gives us a 3x10 matrix (documents * corpus vocab) containing the tf-idf scores.

In [11]:
print(tf_idf.tfidf_matrix.toarray())


[[0.         0.54930615 0.         0.         0.         0.
  0.54930615 0.20273255 0.20273255 0.4054651 ]
 [0.         0.         0.         0.54930615 0.         0.54930615
  0.         0.20273255 0.20273255 0.4054651 ]
 [1.0986123  0.         1.0986123  0.         1.0986123  0.
  0.         0.         0.         0.        ]]


These will serve as the embeddings that we will later compare our queries to, to decide what documents are most relevant to a query.

### Queries

Now that we have the `Document` and the `Retriever` class defined, we can compute the *Term Frequencies* and *Inverse Document Frequencies* for all the documents.

But our goal is to be able to *retrieve* the most relevant *document*, based on a *query*.

So we need a `Query` class for the queries, which we can use to compute the *similarity score* to each `Document` using the `Retriever` class.

Defining the `Query` class.

* Notice that `relevant_document_ids` represents the *ground-truth labels*, which is used to grade the retriever afterwards.

* The datasets that are used in (main.py) come annotated. Each FiQA question ships with the passages that answer it (ground_truths), and each SQuAD question ships with the context paragraph it was written from.


In [12]:
from src.data_utils.query import Query

class DevQuery(Query):
    def __init__(self, text: str, id: str, relevant_document_ids: list[str], label: str):
        
        self.text = text
        self.id = id
        self.relevant_document_ids = relevant_document_ids
        self.label = label

Checking the dummy dataset to mock some GT labels.

In [13]:
for i in range(3):
    print(raw_docs[0]["document_id"][i], raw_docs[0]["text"][i])

d0 the cat sat on the mat
d1 the dog sat on the log
d2 cats and dogs


Instantiating the `Query` class 4 times (as an array), to get a few different example queries

In [14]:
queries = [
    DevQuery(text='is that a giraffe',  id='q0', relevant_document_ids=[''],        label='q0'),    # no common terms with any document
    DevQuery(text='hello dog',          id='q1', relevant_document_ids=['d1'],      label='q1'),    # "dog" in common with d1
    DevQuery(text='cats and dogs',      id='q2', relevant_document_ids=['d2'],      label='q2'),    # all words in common with d2
    DevQuery(text='a dog and a cat',    id='q3', relevant_document_ids=['d0,d1'],   label='q3'),    # "cat" in common with d0, "dog" in common with d1, none in common with d3
]

q0,q1,q2,q3 = queries

Multiple ways to access properties

In [15]:
print(q1.text)          # using the object - preferred (more transparent)
print(q1.getQuery())    # using the method

hello dog
hello dog


We use the methods like this (here used on full array of queries)

In [16]:
[tf_idf.GetQueryVocabulary(q.text) for q in queries]


[['a', 'giraffe', 'is', 'that'],
 ['dog', 'hello'],
 ['and', 'cats', 'dogs'],
 ['a', 'and', 'cat', 'dog']]

`GetQueryVocabulary` is used to compute the query term counts in `GetQueryTermCounts`

In [18]:
[tf_idf.GetQueryTermCounts(q.text) for q in queries]

[{'a': 1, 'giraffe': 1, 'is': 1, 'that': 1},
 {'dog': 1, 'hello': 1},
 {'and': 1, 'cats': 1, 'dogs': 1},
 {'a': 2, 'and': 1, 'cat': 1, 'dog': 1}]

Converting the queries into vectors.

* The query term counts (`GetQueryTermCounts`) can now be used to vectorize the queries using `QueryToVector`.

* The vectors have one entry per word in the corpus vocabulary, and each entry is how many times that word appears in the query.
* This can then later be used to compare the queries to the documents.

In [19]:
print(tf_idf.QueryToVector([q1.text]))
# [[0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (1, 10)>
  Coords	Values
  (0, 3)	1.0


`q1.text` is 'hello dog'

In [21]:
q1.text

'hello dog'

The 1.0 means the word at column 3 (index 4) appears once in the query. It's a raw count, not a TF-IDF weight.

In [32]:
for idx, word in enumerate(tf_idf.GetCorpusVocabulary()):
    print(f'{idx} : {word}')

0 : and
1 : cat
2 : cats
3 : dog
4 : dogs
5 : log
6 : mat
7 : on
8 : sat
9 : the


The word at column 3 is 'dog'.

We can also run `QueryToVector` on all 4 queries

In [39]:
[print(f'{tf_idf.QueryToVector([q.text])} \n ________ \n ') for q in queries]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 0 stored elements and shape (1, 10)> 
 ________ 
 
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (1, 10)>
  Coords	Values
  (0, 3)	1.0 
 ________ 
 
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3 stored elements and shape (1, 10)>
  Coords	Values
  (0, 0)	1.0
  (0, 2)	1.0
  (0, 4)	1.0 
 ________ 
 
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3 stored elements and shape (1, 10)>
  Coords	Values
  (0, 0)	1.0
  (0, 1)	1.0
  (0, 3)	1.0 
 ________ 
 


[None, None, None, None]

This gives us the term count vectors over the entire corpus vocabulary for each Query.

* This tells us how many words in each query are part of the corpus (`DocumentDataset`) vocabulary

* What this does NOT tell us is how well each query overlaps with each document


What we are still missing is calculating the TF-IDF scores between the documents and queries.

* CalculateScores returns a dense NumPy array with one row per query and one column per document. Each entry is how well that document matches that query, and higher means a better match.

* Entry (i, j) is the dot product of query i's count vector with document j's TF-IDF row. That is, for every word the query and document share, it multiplies the word's count in the query by the word's TF-IDF weight in the document, then adds those up.

In [21]:
[tf_idf.CalculateScores([q.getQuery()]) for q in queries]

[array([[0., 0., 0.]], dtype=float32),
 array([[0.        , 0.54930615, 0.        ]], dtype=float32),
 array([[0.      , 0.      , 3.295837]], dtype=float32),
 array([[0.54930615, 0.54930615, 1.0986123 ]], dtype=float32)]

#### BM25


In [28]:
from src.retrievers.bm25 import BM25

class DevBM25(BM25):
    def __init__(self, documents, k1=1.5, b=0.75):
        Retriever.__init__(self, documents)   
        self.k1 = k1
        self.b = b
        
        self.corpus_vocabulary = self.GetCorpusVocabulary()
        self.idf = self.GetInverseDocumentFrequencies()
        self.bm25_matrix = self.GetDocumentsBM25Vectors()

bm25 = DevBM25(raw_docs)

In [24]:
print("Corpus Vocabulary", bm25.GetCorpusVocabulary())
print("BM25 IDF Values", bm25.GetInverseDocumentFrequencies())


doc_lens, avg_len = bm25.GetDocumentLengths()
print("Document Lengths", {getattr(doc, 'id', getattr(doc, '_id', str(doc))): length for doc, length in doc_lens.items()})
print("Average Document Length", avg_len)


print("BM25 Matrix ", bm25.bm25_matrix.shape)

Corpus Vocabulary {'dogs', 'cats', 'cat', 'log', 'sat', 'on', 'dog', 'and', 'the', 'mat'}
BM25 IDF Values {'cat': np.float64(0.9808292530117262), 'sat': np.float64(0.47000362924573563), 'on': np.float64(0.47000362924573563), 'the': np.float64(0.47000362924573563), 'mat': np.float64(0.9808292530117262), 'log': np.float64(0.9808292530117262), 'dog': np.float64(0.9808292530117262), 'dogs': np.float64(0.9808292530117262), 'and': np.float64(0.9808292530117262), 'cats': np.float64(0.9808292530117262)}
Document Lengths {'d0': 6, 'd1': 6, 'd2': 3}
Average Document Length 5.0
BM25 Matrix  (3, 10)


Query test

In [25]:

print("Query Vectors")
print([bm25.QueryToVector([q.getQuery()]).toarray() for q in queries])


for q in queries:
    scores = bm25.CalculateScores([q.getQuery()])
    print(f"Query '{q.id}' ('{q.getQuery()}')")
    print(scores)

Query Vectors
[array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32), array([[0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]], dtype=float32), array([[1., 1., 0., 0., 0., 0., 0., 1., 0., 0.]], dtype=float32), array([[0., 0., 1., 0., 0., 0., 1., 1., 0., 0.]], dtype=float32)]
Query 'q0' ('is that a giraffe')
[[0. 0. 0.]]
Query 'q1' ('hello dog')
[[0.         0.89984334 0.        ]]
Query 'q2' ('cats and dogs')
[[0.     0.     3.5884]]
Query 'q3' ('a dog and a cat')
[[0.89984334 0.89984334 1.1961333 ]]


In [26]:

bm25_created= BM25(raw_docs)

print("BM25 scores:")
query_texts = [q.getQuery() for q in queries]
print(bm25_created.CalculateScores(query_texts))

BM25 scores:
[[0.         0.         0.        ]
 [0.         0.89984334 0.        ]
 [0.         0.         3.5884    ]
 [0.89984334 0.89984334 1.1961333 ]]
